

## Что такое числа с плавающей точкой?

Числа с плавающей точкой — способ **приближённого** представления вещественных чисел в памяти компьютера.  

 
 $$ x = \pm m \times b^e$$
  где  
  - *m* — мантисса (mantissa / significand)  
  - *b* — основание (обычно 2)  
  - *e* — порядок (exponent)

$$ x=(−1)s×1.m_1​m_2​m_3​...m_p​×2^e $$
---

## IEEE 754: стандарт представления

- Основной стандарт для чисел с плавающей точкой  
- Поддерживается всеми современными процессорами и языками программирования


| Компонент | Назначение | Кол-во бит (float32) | Кол-во бит (float64) |
|------------|-------------|----------------------|----------------------|
| Знак (Sign) | 0 = +, 1 = – | 1 | 1 |
| Порядок (Exponent) | | 8 | 11 |
| Мантисса (Fraction) |  | 23 | 52 |
| **Всего** |  | **32 бита** | **64 бита** |

---

##  Форматы чисел с плавающей точкой

| Формат | Название | Размер | Диапазон значений | Пример |
|--------|-----------|---------|------------------|--------|
| `half` | Float16 | 16 бит | ~10⁻⁵ до 10⁵ | ML, GPU |
| `single` | Float32 | 32 бит | ~10⁻³⁸ до 10³⁸ | стандарт в Python `float` |
| `double` | Float64 | 64 бита | ~10⁻³⁰⁸ до 10³⁰⁸ | научные вычисления |
| `extended` | Float80 / Float128 | 80–128 бит | ещё шире | редко, high-precision math |
| `bfloat16` | Brain float | 16 бит (8 exp + 7 mant) | низкая точность | Deep Learning |

---

Например, `0.15625` в двоичном виде  = 0.00101₂

##  Особые значения IEEE 754

| Значение | Биты | Комментарий |
|-----------|------|-------------|
| +0, −0 | exp=0, mant=0 | два нуля, но разные знаки |
| +inf, −inf | exp все 1, mant = 0 | результат переполнения |
| NaN | exp все 1, mant != 0 | “Not a Number” (например, 0/0) |
| Denormals | exp=0, mant != 0 | субнормальные числа (около нуля) |


## Основные источники ошибок

1. **Ограниченная точность**
   - Не все дроби представимы в двоичной форме (например, 0.1)
   - Ошибки округления накапливаются

2. **Нестабильность алгоритмов**
   - Разность близких чисел → потеря значимых разрядов (catastrophic cancellation)

3. **Ассоциативность не работает**
   - `(a + b) + c != a + (b + c)`

4. **Погрешности при сравнении**
   - `x == y` ненадёжно  
   - лучше использовать `|x - y| < ε`


## Суммирование

In [ ]:
numbers = [1.0, 1e16, -1e16]

s = 0.0
for x in numbers:
    s += x

print(s)

0.0


In [37]:
a = float("nan")
a + 0.5

nan

### Алгоритм суммирования Кэхэна (Kahan Summation)

Идея: Компенсировать потерянные младшие биты с помощью дополнительной переменной

In [21]:
s = 0.
compensation = 0.

for x in numbers:
    y = x - compensation
    t = s + y
    compensation = (t - s) - y
    s = t

compensation

0.0

In [ ]:
from decimal import Decimal, localcontext, getcontext

getcontext().prec = 100

with localcontext(prec=100) as ctx:
    a = Decimal("0.1")
    b = Decimal("0.2")
    c = a + b
    
print(c)

0.3


In [44]:
a = 5 ** 1
b = 5 ** 1
id(a), id(b)

(10919624, 10919624)